# Geração de Dados

In [271]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean, RollingStd
import lightgbm as lgb
import holidays
from scipy.stats import nbinom
from scipy.optimize import minimize

In [272]:

def generate_m5_simulated_data(n_stores=3, n_skus=5, start_date="2023-01-01", days=365, seed=42):
    """
    Gera um dataset simulando a dinâmica da competição M5 da Walmart:
    - Vendas de contagem (Poisson/Binomial Negativa) com zeros (intermitência).
    - Variações de preço com elasticidade-preço.
    - Sazonalidade semanal e eventos especiais.
    """
    np.random.seed(seed)
    dates = pd.date_range(start=start_date, periods=days, freq="D")
    n_days = len(dates)
    
    data_list = []

    for store_id in range(1, n_stores + 1):
        store_name = f"STORE_{store_id:02d}"
        for sku_id in range(1, n_skus + 1):
            item_id = f"FOODS_1_{sku_id:03d}"
            
            # Demanda base do SKU na loja
            base_demand = np.random.uniform(0.5, 5.0)
            
            # Fator de sazonalidade semanal (fds vende mais)
            dow_factor = np.tile([0.8, 0.85, 0.9, 0.95, 1.1, 1.4, 1.3], int(np.ceil(n_days/7)))[:n_days]
            
            # Variação e promoção de preço
            base_price = np.random.uniform(2.0, 15.0)
            prices = base_price * np.random.choice([1.0, 0.85, 0.70], size=n_days, p=[0.8, 0.15, 0.05])
            price_elasticity = np.exp(-0.15 * (prices - base_price))
            
            # Dias com eventos/promoções especiais
            event_indices = np.random.choice(n_days, size=12, replace=False)
            event_impact = np.ones(n_days)
            event_impact[event_indices] = np.random.uniform(1.3, 2.2, size=12)
            
            # Parâmetro lambda da demanda diária
            lambda_t = base_demand * dow_factor * price_elasticity * event_impact
            
            # Vendas reais observadas (intermitentes)
            sales = np.random.poisson(lambda_t)
            
            for d_idx, d_date in enumerate(dates):
                data_list.append({
                    'date': d_date,
                    'store_id': store_name,
                    'item_id': item_id,
                    'sales': sales[d_idx],
                    'sell_price': round(prices[d_idx], 2),
                    'is_event': 1 if d_idx in event_indices else 0,
                })

    return pd.DataFrame(data_list)

df_raw = generate_m5_simulated_data(n_stores=3, n_skus=5, days=365) # Função criada anteriormente

df_raw['unique_id'] = df_raw['store_id'] + '_' + df_raw['item_id']

df_nixtla = df_raw.rename(columns={
    'date': 'ds',
    'sales': 'y'
})

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import nbinom
from scipy.optimize import minimize
from mlforecast import MLForecast


class ISSMForecast:
    """Intermittent State-Space Model (ISSM) wrapper using MLForecast and Negative Binomial distribution.

    This model isolates the conditional expectation (lambda_t) using standard MLForecast regressors
    and fits a per-series dispersion parameter (r) via Maximum Likelihood Estimation (MLE).
    It projects inventory quantiles via the inverse CDF (PPF) of the Negative Binomial distribution.

    Demand Regime Applicability
    ---------------------------
    - **Intermittent Demand (Zero-Inflated)**: Highly recommended. Handles frequent zeros efficiently.
    - **Erratic / Lumpy Demand**: Highly recommended. Captures high variance (variance > mean) via 'r'.
    - **Smooth / Low-Variance Demand**: Supported. As r increases, it naturally converges to a Poisson regime.
    - **Continuous / High-Volume Demand**: Not recommended.

    References & Architecture
    -------------------------
    - **Theoretical Framework**: Inspired by Lokad's white-boxed ISSM approach for intermittent 
      demand and zero-inflated sales (M5 Forecasting Competition, 2020).
      Paper reference: "A white-boxed ISSM approach to estimate uncertainty distributions of Walmart sales".
    - **Implementation Variant**: Unlike the pure state-space formulation (which uses ETS(A,N,M) state updates 
      and Monte Carlo simulation), this wrapper decouples expectation from dispersion:
        1. Employs `MLForecast` regressors (e.g., LightGBM) to forecast conditional expectation (lambda_t).
        2. Fits a per-series dispersion parameter (r) via Negative Binomial MLE on in-sample residuals.
        3. Derives discrete inventory quantiles via inverse CDF (PPF) projection.

    Parameters
    ----------
    fcst : MLForecast
        An un-fitted or fitted MLForecast instance configured with a single underlying regressor.
    """

    def __init__(self, fcst: MLForecast):
        self.fcst = fcst
        self.r_dict = {}
        self.r_fallback = 1.0

    @property
    def model(self):
        """Extracts and validates the underlying trained estimator from the MLForecast container.

        Returns
        -------
        object
            The single regression estimator stored inside MLForecast.

        Raises
        ------
        ValueError
            If MLForecast has not been fitted or contains multiple estimators.
        """
        if not hasattr(self.fcst, "models_") or not self.fcst.models_:
            raise ValueError("The MLForecast object has not been fitted yet.")

        if len(self.fcst.models_) > 1:
            raise ValueError(
                f"ISSMForecast supports exactly 1 model, but found: {list(self.fcst.models_.keys())}"
            )

        return next(iter(self.fcst.models_.values()))

    def _nbinom_log_likelihood(self, params: list, y: np.ndarray, lambda_t: np.ndarray) -> float:
        """Computes the negative log-likelihood of the Negative Binomial distribution.

        Process
        -------
        1. Ensures parameter r remains positive.
        2. Converts (lambda_t, r) to the success probability parameter `p = r / (r + lambda_t)`.
        3. Computes the log probability mass function (logpmf) for discrete counts.
        4. Applies protection against negative infinity underflows (-1e2 floor).

        Parameters
        ----------
        params : list of float
            Parameter vector containing [r] (dispersion/size parameter of the Negative Binomial).
        y : numpy.ndarray
            Observed historical demand values.
        lambda_t : numpy.ndarray
            In-sample predicted conditional expectation (lambda).

        Returns
        -------
        float
            Negative log-likelihood value to be minimized.
        """
        r = params[0]
        if r <= 0:
            return 1e10

        lambda_t = np.maximum(lambda_t, 1e-6)
        p = r / (r + lambda_t)

        log_pdf = nbinom.logpmf(y, r, p)
        log_pdf = np.nan_to_num(log_pdf, neginf=-1e2)

        return -np.sum(log_pdf)

    def _estimate_r(self, y_obs: np.ndarray, lambdas: np.ndarray) -> float:
        """Internal helper to estimate the dispersion parameter (r) via Maximum Likelihood Estimation.

        Process
        -------
        Uses L-BFGS-B optimization to minimize `_nbinom_log_likelihood` over the observed target
        series and in-sample predictions. Parameter r is bounded between 1e-3 and 50.0 to prevent
        collapse into thin-tailed distributions (Poisson) and ensure tail coverage.

        Parameters
        ----------
        y_obs : numpy.ndarray
            Target demand values for a specific series.
        lambdas : numpy.ndarray
            Predicted conditional expectation values (lambda_t) for the same series.

        Returns
        -------
        float
            Optimized per-series dispersion parameter r.
        """
        res = minimize(
            self._nbinom_log_likelihood,
            x0=[1.0],
            method='L-BFGS-B',
            args=(y_obs, lambdas),
            bounds=[(1e-3, 50.0)]
        )
        return float(res.x[0])

    def _compute_time_decay_weights(self, df: pd.DataFrame, time_col: str = 'ds', gamma: float = 0.5) -> np.ndarray:
        """Generates an exponential time-decay weight vector based on date recency.

        The gamma scale is ANNUAL (e.g., gamma=0.5 reduces weight to ~60% after 1 year),
        regardless of series frequency (daily, weekly, monthly, or hourly).

        Parameters
        ----------
        gamma : float, optional
            Exponential decay parameter for recency weighting (scale is annual).
            If provided, applies larger weights to recent historical samples during model training.
        """

        dates = pd.to_datetime(df[time_col])
        max_date = dates.max()

        seconds_in_year = 365.25 * 86400.0
        delta_years = (max_date - dates).dt.total_seconds() / seconds_in_year

        weights = np.exp(-gamma * delta_years).values
        return weights

    def fit(self, 
            df_train: pd.DataFrame, 
            id_col: str = 'unique_id', 
            time_col: str = 'ds', 
            target_col: str = 'y', 
            static_features: list = None,
            gamma: float = None) -> "ISSMForecast":
        """Fits the underlying MLForecast model and optimizes per-series dispersion parameters.

        Process
        -------
        1. Fits the MLForecast pipeline on the training dataset.
        2. Preprocesses the data to generate feature matrices.
        3. Generates in-sample point predictions (lambda_t).
        4. Iterates over each unique ID and executes `_estimate_r` via MLE.
        5. Computes `r_fallback` as the global median of estimated 'r' values for cold-start prediction.

        Parameters
        ----------
        df_train : pandas.DataFrame
            Training data containing series IDs, timestamps, target values, and features.
        id_col : str, default='unique_id'
            Column name identifying individual time series.
        time_col : str, default='ds'
            Column name containing timestamps.
        target_col : str, default='y'
            Column name for target demand variable.
        static_features : list of str, optional
            List of static feature names to preserve.

        Returns
        -------
        self : ISSMForecast
            Fitted instance of the ISSMForecast class.
        """
        self.id_col = id_col
        self.time_col = time_col
        self.target_col = target_col
        static_features = static_features or []

        df_fit = df_train.copy()

        fit_kwargs = {}
        if gamma is not None:
            weights_array = self._compute_time_decay_weights(df_fit, time_col=time_col, gamma=gamma)
            df_fit["_temp_weight"] = weights_array
            fit_kwargs["weight_col"] = "_temp_weight"

        self.fcst.fit(
            df_fit,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            static_features=static_features,
            **fit_kwargs,
        )

        df_prep = self.fcst.preprocess(
            df_train,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            static_features=static_features,
        )

        self.exog_cols_ = self.fcst.ts.features_order_
        X_train = df_prep[self.exog_cols_]
        df_prep['lambda_t'] = self.model.predict(X_train)

        # Fit dispersion parameter per series via MLE
        self.r_dict = {}
        for uid, group in df_prep.groupby(id_col):
            y_obs = group[target_col].values
            lambdas = group['lambda_t'].values
            self.r_dict[uid] = self._estimate_r(y_obs, lambdas)

        # Fallback value for unseen series during inference
        self.r_fallback = float(np.median(list(self.r_dict.values())))

        return self
    
    def predict(self, h: int, X_df: pd.DataFrame = None, quantiles: list = [0.50, 0.67, 0.95, 0.99]) -> pd.DataFrame:
        """Generates out-of-sample probabilistic forecast quantiles using the Negative Binomial CDF.

        Process
        -------
        1. Executes recursive forecasting via MLForecast to obtain future lambda_t values.
        2. Maps historical `r` dispersion parameters (or applies `r_fallback` for new series).
        3. Computes exact integer quantiles using the Negative Binomial Percent Point Function (PPF/Inverse CDF)
           and applies `np.ceil` to ensure discrete inventory units.

        Parameters
        ----------
        h : int
            Forecast horizon.
        X_df : pandas.DataFrame, optional
            Exogenous features for the forecast horizon.
        quantiles : list of float, default=[0.50, 0.67, 0.95, 0.99]
            Target quantile values.

        Returns
        -------
        df_pred : pandas.DataFrame
            DataFrame containing time series identifiers, predicted lambda_t, dispersion parameters,
            and discrete stock quantile estimates (`q_*`).
        """

        df_pred = self.fcst.predict(h=h, X_df=X_df)

        model_key = next(iter(self.fcst.models_.keys()))
        df_pred = df_pred.rename(columns={model_key: 'lambda_t'})
        df_pred['r_dispersion'] = df_pred[self.id_col].map(self.r_dict).fillna(self.r_fallback)

        for q in sorted(quantiles):
            col_name = f'q_{int(q * 100)}'
            p_param = df_pred['r_dispersion'] / (df_pred['r_dispersion'] + df_pred['lambda_t'])
            df_pred[col_name] = np.ceil(nbinom.ppf(q, df_pred['r_dispersion'], p_param))

        return df_pred

In [274]:
def temporal_split_by_horizon(df, time_col='ds', horizon_days=28):
    """
    Separa estritamente o DataFrame por data de corte.
    Os últimos 'horizon_days' viram o conjunto de TESTE/VALIDAÇÃO.
    """
    df = df.sort_values(time_col)
    max_date = df[time_col].max()
    cutoff_date = max_date - pd.Timedelta(days=horizon_days)
    
    df_train = df[df[time_col] <= cutoff_date].copy()
    df_test = df[df[time_col] > cutoff_date].copy()
    
    return df_train, df_test, cutoff_date

In [275]:
us_holidays = holidays.US(years=range(2022, 2027))

_extended_holiday_dates = set()
for h_date in us_holidays.keys():
    h_timestamp = pd.Timestamp(h_date)
    for offset in range(-2, 1):  # Véspera (-2, -1) e o próprio dia (0)
        _extended_holiday_dates.add((h_timestamp + pd.Timedelta(days=offset)).date())


def is_holiday_window(dates) -> pd.Series:
    """Retorna 1 para o dia do feriado e os 2 dias que o antecedem."""
    dates_series = pd.Series(dates)
    return dates_series.dt.date.isin(_extended_holiday_dates).astype(int)

def add_relative_price(df: pd.DataFrame, id_col: str = 'unique_id', price_col: str = 'sell_price') -> pd.DataFrame:
    """Calcula o preço relativo de cada SKU em relação ao seu preço médio histórico."""
    df = df.copy()
    
    # Preço médio histórico por SKU
    mean_price_per_sku = df.groupby(id_col)[price_col].transform('mean')
    
    # Preço Relativo (ex: 0.85 significa 15% de desconto em relação à média do SKU)
    df['relative_price'] = df[price_col] / (mean_price_per_sku + 1e-6)
    
    return df

df_nixtla['is_holiday_window'] = is_holiday_window(df_nixtla['ds'])
df_nixtla = add_relative_price(df_nixtla)

fcst = MLForecast(
    models={
        'lgb_issm': lgb.LGBMRegressor(
            objective='poisson',
            metric='rmse',
            n_estimators=100,
            learning_rate=0.05,
            random_state=42,
            verbosity=-1
        )
    },
    freq='D',
    lags=[7, 14, 28],
    lag_transforms={
        1: [RollingMean(window_size=7), RollingStd(window_size=7)],
    },
    date_features=['dayofweek', 'month', 'dayofyear']
)

In [276]:
df_train, df_test, cutoff = temporal_split_by_horizon(df_nixtla, horizon_days=28)

issm = ISSMForecast(fcst)
issm.fit(df_train[["ds", "y", "sell_price", 'relative_price', "is_event", "is_holiday_window", "unique_id"]], gamma=0.7)

In [277]:
dias_por_sku = df_test[["ds", "sell_price", "is_event", "is_holiday_window", "unique_id"]].groupby("unique_id")["ds"].nunique()
print(dias_por_sku)

unique_id
STORE_01_FOODS_1_001    28
STORE_01_FOODS_1_002    28
STORE_01_FOODS_1_003    28
STORE_01_FOODS_1_004    28
STORE_01_FOODS_1_005    28
STORE_02_FOODS_1_001    28
STORE_02_FOODS_1_002    28
STORE_02_FOODS_1_003    28
STORE_02_FOODS_1_004    28
STORE_02_FOODS_1_005    28
STORE_03_FOODS_1_001    28
STORE_03_FOODS_1_002    28
STORE_03_FOODS_1_003    28
STORE_03_FOODS_1_004    28
STORE_03_FOODS_1_005    28
Name: ds, dtype: int64


In [278]:
df_res = issm.predict(h=28, X_df=df_test[["ds", "sell_price", "relative_price", "is_event", "is_holiday_window", "unique_id"]], quantiles=[0.95])
df_res.loc[:, "y"] = df_test["y"].values

## Avaliação

In [279]:
import numpy as np
import pandas as pd


class ISSMForecastEvaluator:
    """Evaluator utility for probabilistic quantile forecasts."""

    @staticmethod
    def pinball_loss(y_true: np.ndarray, y_pred: np.ndarray, quantile: float) -> float:
        """Computes the Pinball Loss (Quantile Loss) for a given target quantile."""
        y_true = np.asarray(y_true)
        y_pred = np.asarray(y_pred)
        
        # Remove NaNs to prevent metric corruption
        mask = ~(np.isnan(y_true) | np.isnan(y_pred))
        y_true_clean = y_true[mask]
        y_pred_clean = y_pred[mask]

        err = y_true_clean - y_pred_clean
        return float(np.maximum(quantile * err, (quantile - 1) * err).mean())

    @classmethod
    def evaluate(
        cls, 
        df_res: pd.DataFrame, 
        target_col: str = 'y', 
        quantis: list = [0.50, 0.67, 0.95, 0.99]
    ) -> pd.DataFrame:
        """Evaluates empirical coverage and quantile loss over out-of-sample backtest predictions.

        Parameters
        ----------
        df_res : pandas.DataFrame
            DataFrame containing real ground truth targets and forecasted quantile columns (`q_*`).
        target_col : str, default='y'
            Name of the column containing real observed values.
        quantis : list of float, default=[0.50, 0.67, 0.95, 0.99]
            List of target quantiles evaluated.

        Returns
        -------
        pandas.DataFrame
            Summary dataframe containing Pinball Loss, Target Coverage, and Empirical Coverage.
        """
        results = {}

        if target_col not in df_res.columns:
            raise KeyError(f"Target column '{target_col}' not found in the input DataFrame.")

        for q in sorted(quantis):
            col_q = f'q_{int(q * 100)}'
            if col_q not in df_res.columns:
                continue

            loss = cls.pinball_loss(
                y_true=df_res[target_col].values,
                y_pred=df_res[col_q].values,
                quantile=q
            )
            
            empirical_coverage = (df_res[target_col] <= df_res[col_q]).mean()

            results[col_q] = {
                'Pinball Loss': round(loss, 4),
                'Target Coverage': f"{int(q * 100)}%",
                'Empirical Coverage': f"{empirical_coverage * 100:.1f}%"
            }

        return pd.DataFrame(results).T

In [280]:
ISSMForecastEvaluator.evaluate(df_res)

,Pinball Loss,Target Coverage,Empirical Coverage
q_95,0.5005,95%,85.5%
